In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Flatten, Dense, BatchNormalization
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import Model

###**Dataset link:** https://www.kaggle.com/datasets/ted8080/house-prices-and-images-socal/data

##**Extract the data file**

In [3]:
import zipfile
zip_path='/content/data.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/house_data")

In [ ]:
l=[i for i in os.listdir('/content/house_data/socal2/socal_pics')]
print(len(l))

15330


In [4]:
data=pd.read_csv('/content/house_data/socal2.csv')
df1=data[['image_id', 'price']]
print(df1.shape)
df1.head()


(15474, 2)


,image_id,price
0,0,201900
1,1,228500
2,2,273950
3,3,350000
4,4,385100


##**Create label**

In [5]:
import os
import cv2
image=[]
label=[]
path='/content/house_data/socal2/socal_pics'
for img in os.listdir(path):
  label.append(int(img.split('.')[0]))
  image.append(img)
print((image))
print((label))


['1942.jpg', '1934.jpg', '2191.jpg', '5362.jpg', '11758.jpg', '13414.jpg', '2000.jpg', '13652.jpg', '1977.jpg', '9562.jpg', '14592.jpg', '12991.jpg', '5586.jpg', '92.jpg', '9575.jpg', '13422.jpg', '13348.jpg', '6487.jpg', '2464.jpg', '8764.jpg', '11391.jpg', '5523.jpg', '2458.jpg', '8441.jpg', '1350.jpg', '3787.jpg', '11051.jpg', '1479.jpg', '5941.jpg', '14876.jpg', '13614.jpg', '583.jpg', '475.jpg', '13201.jpg', '1573.jpg', '8095.jpg', '11865.jpg', '5335.jpg', '6242.jpg', '727.jpg', '14956.jpg', '11764.jpg', '9236.jpg', '6039.jpg', '4691.jpg', '11371.jpg', '13000.jpg', '274.jpg', '3933.jpg', '8695.jpg', '13850.jpg', '9121.jpg', '8810.jpg', '15044.jpg', '8847.jpg', '13394.jpg', '8513.jpg', '4807.jpg', '12355.jpg', '14928.jpg', '15383.jpg', '9877.jpg', '14721.jpg', '10675.jpg', '8754.jpg', '5495.jpg', '14954.jpg', '4763.jpg', '3081.jpg', '1960.jpg', '13831.jpg', '3096.jpg', '14405.jpg', '12097.jpg', '6761.jpg', '6754.jpg', '10186.jpg', '1871.jpg', '1849.jpg', '11690.jpg', '4368.jpg', '2

In [6]:
df2=df1[df1['image_id'].isin(label)]
df2=df2.reset_index(drop=True)
df2.head()

,image_id,price
0,2,273950
1,3,350000
2,4,385100
3,5,350000
4,6,415000


In [7]:
df3=pd.DataFrame({'img_id':label, 'Image':image})
df3=df3.sort_values(by='img_id')
df3=df3.reset_index(drop=True)
df3.head()

,img_id,Image
0,2,2.jpg
1,3,3.jpg
2,4,4.jpg
3,5,5.jpg
4,6,6.jpg


##**Finale Data**

In [8]:
df=df3
df['price']=df2['price']
print(df.shape)
df.head(5)

(15330, 3)


,img_id,Image,price
0,2,2.jpg,273950
1,3,3.jpg,350000
2,4,4.jpg,385100
3,5,5.jpg,350000
4,6,6.jpg,415000


In [16]:
df['price']=np.log(df['price'])
df

,img_id,Image,price
0,2,2.jpg,2.527383
1,3,3.jpg,2.546761
2,4,4.jpg,2.554220
3,5,5.jpg,2.546761
4,6,6.jpg,2.560017
...,...,...,...
15325,15469,15469.jpg,2.621996
15326,15470,15470.jpg,2.622065
15327,15471,15471.jpg,2.619738
15328,15472,15472.jpg,2.625574


##**Train Test Split**

In [17]:
train_df=df.sample(frac=1, random_state=0).iloc[0:200]
test_df=df.sample(frac=1, random_state=0).iloc[201:300]

In [18]:
print('shape of train_df', train_df.shape)
print('shape of test_df', test_df.shape)

shape of train_df (200, 3)
shape of test_df (99, 3)


##**Data Augumentation**

In [19]:
train_generator=ImageDataGenerator(
    rescale=1./255 ,
    rotation_range=30 ,
    zoom_range=0.2 ,
    shear_range=0.2 ,
    width_shift_range=0.2 ,
    height_shift_range=0.2 ,
    horizontal_flip=True
)

val_generator=ImageDataGenerator(
    rescale=1./255 ,
    rotation_range=30,
    shear_range=0.2,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)


train_ds=train_generator.flow_from_dataframe(
    train_df,
    directory=path ,
    x_col='Image',
    y_col=['price'] ,
    target_size=(200,200) ,
    class_mode='raw'
)

val_ds=val_generator.flow_from_dataframe(
    test_df ,
    directory=path,
    x_col='Image' ,
    y_col=['price'] ,
    target_size=(200,200),
    class_mode='raw'

)

Found 200 validated image filenames.
Found 99 validated image filenames.


In [20]:
for images, labels in train_ds.take(1):
  print(images, labels)

AttributeError: 'DataFrameIterator' object has no attribute 'take'

In [21]:
print(type(train_ds))
print(type(val_ds))

<class 'keras.src.legacy.preprocessing.image.DataFrameIterator'>
<class 'keras.src.legacy.preprocessing.image.DataFrameIterator'>


##**Train the Model**

In [22]:
resnet=ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

res_output=resnet.output

flat=Flatten()(res_output)

batch1=BatchNormalization()(flat)

dense1=Dense(128, activation='relu')(batch1)

batch2=BatchNormalization()(dense1)

dense2=Dense(128, activation='relu')(batch2)

batch3=BatchNormalization()(dense2)

dense3=Dense(64, activation='relu')(batch3)

batch4=BatchNormalization()(dense3)

output1=Dense(1, activation='linear', name='price')(batch4)

model=Model(inputs=resnet.input, outputs=output1)

##**Compile and Evaluate the mode**

In [23]:
model.compile(optimizer='adam', loss='mae', metrics=['mae'])

model.fit(train_ds, epochs=20, validation_data=val_ds)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 76s 9s/step - loss: 2.5878 - mae: 2.5878 - val_loss: 0.6234 - val_mae: 0.6234
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 56s 8s/step - loss: 2.4848 - mae: 2.4848 - val_loss: 1.2713 - val_mae: 1.2713
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 55s 8s/step - loss: 2.4021 - mae: 2.4021 - val_loss: 1.5498 - val_mae: 1.5498
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 57s 9s/step - loss: 2.3200 - mae: 2.3200 - val_loss: 1.7449 - val_mae: 1.7449
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 55s 9s/step - loss: 2.2346 - mae: 2.2346 - val_loss: 2.6913 - val_mae: 2.6913
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 55s 8s/step - loss: 2.1490 - mae: 2.1490 - val_loss: 2.6879 - val_mae: 2.6879
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 55s 8s/step - loss: 2.0626 - mae: 2.0626 - val_loss: 1.1787 - val_mae: 1.1787
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 55s 9s/step - loss: 1.9576 - mae: 1.9576 - val_loss: 0.6216 - val_mae: 0.6216
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 54s 8s/step - loss: 1.8526 - mae: 1.8526 - val_loss: